# 20 — Hurtigtest av ukentlig nivåkorreksjon etter notebook 19

**Henrik Remmen (541747), gruppe 119 — [119] SINTEF Databreach**

Notebook 19 og `hydro_lp_v6.csv` (rapportert public AUC **0.99571**) er den frosne hovedkandidaten.
Denne notebooken er et **lokalt, utforskende forskningsforsøk**, ikke en ny innleveringsnotebook.

**Originale CSV-er og artifacts er read-only.** Alle nye resultater havner i en unik mappe under
`artifacts/weekly_correction_probe/`. Ingen submission skrives eller overskrives.
Notebook 19 importeres/kjøres ikke. Ingen XGBoost-/LightGBM-fulltrening eller nye LP-løsninger.

### Hypotese og begrensning
Kan en liten modell for ukens forventede ON-andel forbedre de lagrede LP-modellprediksjonene?
Vi trener per generator en regularisert lineær modell på **ukentlig nivåfeil**, og legger til
et begrenset, konstant logit-skift innen hver uke. Dette beholder rangeringen av timer
innen den enkelte uke/generator: **det kan ikke reparere feil timing innen uken**.
Micro-AUC kan likevel endres fordi rangeringen mellom uker/generatorer endres.

To matchede korreksjonsmodeller testes mot uendret LP-blend:
- **control:** sammendrag av eksisterende LP-blend- og v5-prediksjoner.
- **enhanced:** samme input + LP-ukesammendrag, minimumseffekt-proxy og lønnsomme perioder.

Enkle margin-, run-length- og start/stopp-features finnes allerede i v5
(`sintef_research_features.py`, `sintef_path_features.py`). Vi hevder ikke at alle signalene
er nye: testen må vise ekstra verdi **utover den lagrede modellen og korreksjonskontrollen**.

Fast oppsett: 2020 → 2021 som ekstra robusthetssjekk, og 2021 → 2022 som hovedprobe.
De siste seks treningscasene før hvert nytt år fjernes, fordi de dekker timer i neste år.
Alle disse årene har vært analysert tidligere: resultatene er **ikke urørt holdout-evidens**.


In [1]:
# KJØR FØRST — ressurser og små, faste korreksjoner; ingen parameterjakt.
import os
RUN_PROFILE = "local"       # Dette er en forskningsnotebook med artifact-avhengigheter.
LOCAL_CPU_THREADS = 4      # Små lineære modeller: flere tråder gir ofte ingen gevinst.
THREADS = max(1, min(LOCAL_CPU_THREADS if RUN_PROFILE == "local" else 4, os.cpu_count() or 1, 4))
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = str(THREADS)

RUN_SMALL_MODELS = True    # False: bare innlasting, feilanalyse og feature-konstruksjon.
RUN_EARLY_CHECK = True     # 2020 → 2021 i tillegg til 2021 → 2022.
RIDGE_ALPHA = 100.0
SHRINKAGE = 0.25           # Bruk bare 25 % av den lærte nivåkorreksjonen.
MAX_LEVEL_CHANGE = 0.10    # Maks foreslått endring i ukens ON-andel.
MAX_LOGIT_SHIFT = 1.5      # Ekstra grense; målet kan bli uoppnåelig ved denne grensen.
print(f"{THREADS} CPU-tråder. Ingen fulltrening, ingen LP-beregning, ingen submission.")


4 CPU-tråder. Ingen fulltrening, ingen LP-beregning, ingen submission.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import sys
import time
import uuid
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/sintef_weekly_probe.py").is_file()
             and (p / "data/kernel").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Kjør notebooken i prosjektet med src/, data/ og resultatene fra 19.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.sintef_weekly_probe import (
    file_digest, load_probe_inputs, weekly_features, purged_training_mask,
    error_diagnostics, run_probe, score_predictions,
)

# Fingeravtrykk av eksisterende CSV-er før forsøket. Kun lesing.
protected_paths = sorted(set(
    list((ROOT / "data").rglob("*.csv"))
    + list((ROOT / "submissions").rglob("*.csv"))
    + list((ROOT / "artifacts/hydro_lp_v6").rglob("*.csv"))
    + list(ROOT.glob("*.csv"))
))
ORIGINAL_CSV_HASHES = {str(p.relative_to(ROOT)): file_digest(p) for p in protected_paths}
STARTED = time.perf_counter()
bundles, units, plants, topology, provenance = load_probe_inputs(ROOT)
print("Valideringsprediksjoner, rådata-identitet og eksakt LP-cache kontrollert.")
print("Ingen 334-feature-matrise eller fullmodell ble bygget.")
display(pd.DataFrame([
    dict(year=y, cases=len(b["case_ids"]), generators=len(units), hours=b["base"].shape[-1])
    for y, b in bundles.items()
]))


Valideringsprediksjoner, rådata-identitet og eksakt LP-cache kontrollert.
Ingen 334-feature-matrise eller fullmodell ble bygget.


,year,cases,generators,hours
0,2020,366,14,168
1,2021,365,14,168
2,2022,365,14,168


## 1. Purge, feilanalyse og egen resultatmappe — ingen trening

Fasit brukes bare til evaluering/diagnostikk og senere som treningsmål på treningsåret.
«Nær overgang» betyr timene på hver side av en faktisk ON/OFF-endring innen uken;
dette er en **diagnostisk gruppe**, ikke en feature.

Alle skriveruter under er under `OUT`, en ny mappe som ikke kan finnes fra før.
Det lagres ikke noe i originalmappene.


In [3]:
pairs = [(2020, 2021), (2021, 2022)] if RUN_EARLY_CHECK else [(2021, 2022)]
split_rows = []
for train_year, valid_year in pairs:
    tr, va = bundles[train_year], bundles[valid_year]
    keep = purged_training_mask(tr["starts"], va["starts"])
    split_rows.append(dict(train_year=train_year, validation_year=valid_year,
        train_weeks=int(keep.sum()), purged_weeks=int((~keep).sum()),
        latest_training_hour=str((tr["starts"][keep] + pd.Timedelta(hours=167)).max()),
        first_validation_hour=str(va["starts"].min())))
display(pd.DataFrame(split_rows))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUT = ROOT / "artifacts/weekly_correction_probe" / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

diagnostics = pd.concat([
    error_diagnostics(b, units).assign(year=y) for y,b in bundles.items()
], ignore_index=True)
diagnostics.to_csv(OUT / "original_error_groups.csv", index=False)
display(diagnostics.query("year == 2022").pivot(index="unit", columns="group", values="brier").round(5))
print("Brier: lavere er bedre. Gruppene overlapper og har ulik vanskelighetsgrad.")
print("Nye resultater:", OUT)


,train_year,validation_year,train_weeks,purged_weeks,latest_training_hour,first_validation_hour
0,2020,2021,360,6,2020-12-31 23:00:00+00:00,2021-01-01 00:00:00+00:00
1,2021,2022,359,6,2021-12-31 23:00:00+00:00,2022-01-01 00:00:00+00:00


group,away_from_transition,constant_weeks,mixed_weeks,near_true_transition
unit,,,,
Byrte_G1,0.00444,0.00026,0.01462,0.07025
Haukeli_G1,0.04277,0.01678,0.08464,0.25440
Hogga_G1,0.04061,0.00293,0.07902,0.20844
Kjela_G1,0.03036,0.00832,0.04894,0.12869
Lio_G1,0.05857,0.08822,0.05635,0.19223
Songa_G1,0.00508,0.00002,0.01791,0.09249
Tokke_G1,0.00816,0.00012,0.02590,0.14517
Tokke_G2,0.01624,0.00081,0.04335,0.19978
Tokke_G3,0.00928,0.00042,0.02966,0.16886


Brier: lavere er bedre. Gruppene overlapper og har ulik vanskelighetsgrad.
Nye resultater: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/weekly_correction_probe/20260928T233806Z_3aa70e0a


## 2. Små feature-matriser — ingen trening

Features bygges uten å lese fasit:
- Ukentlig gjennomsnitt, spredning, usikkerhet og uenighet mellom lagrede modellprediksjoner.
- LP-planlagt vannføring og variasjon over uken.
- Om en **tilnærmet, kapasitet-proporsjonal** fordeling av LP-effekt ligger under generatorens minimum.
- Sammenhengende perioder med positiv margin relativt til LP-ens vannkostnadsterskel.
- Grov lønnsomhet for disse periodene etter start-/stoppkostnad.
- Variasjon i LP-beregnet magasinvolum — **ikke observert framtidig magasinvolum**.

Effektfordelingen er ikke SHOPs generatorfordeling. Lønnsomhetsberegningen er ikke eksakt SHOP-profitt.
Disse størrelsene brukes som myke proxy-features, aldri som regler som tvinger en generator av/på.
Hele prisbanen er oppgitt input til planleggingscaset; vi predikerer ikke framtidig strømpris.


In [4]:
features = {year: weekly_features(b, units, plants, topology) for year,b in bundles.items()}
feature_rows = []
for arm, unit_frames in features[2021].items():
    for unit, frame in unit_frames.items():
        feature_rows.extend(dict(arm=arm, unit=unit, feature=f) for f in frame.columns)
pd.DataFrame(feature_rows).to_csv(OUT / "feature_manifest.csv", index=False)
display(pd.DataFrame([
    dict(arm=arm, features=len(frames[units[0]].columns), training_rows_per_generator=len(bundles[2021]["case_ids"])-6)
    for arm, frames in features[2021].items()
]))


,arm,features,training_rows_per_generator
0,control,10,359
1,enhanced,29,359


## 3. KUN SMÅ MODELLER — ikke boosting

Maks 56 små Ridge-modeller når begge årssjekkene er på, hver med cirka 360 treningsrader.
Standardisering tilpasses bare treningsukene etter purge. Kontroll og enhanced har samme
regularisering, demping og korreksjonsgrenser. Ingen parametere velges fra 2022.

Dette er ment å ta kort tid, men kjøretiden er ikke målt på din maskin.
Metrikkberegning på millioner av timeprediksjoner kan ta lengre tid enn selve Ridge-treningen.
Ingen bootstrap eller stort parametersøk kjøres.


In [5]:
all_scores, fit_logs, coefficient_tables = [], [], []
probe_predictions = {}
if RUN_SMALL_MODELS:
    for train_year, valid_year in pairs:
        t0 = time.perf_counter()
        tr, va = bundles[train_year], bundles[valid_year]
        corrected, fit_log, coefs = run_probe(
            tr, va, features[train_year], features[valid_year], units,
            alpha=RIDGE_ALPHA, shrinkage=SHRINKAGE,
            max_level_change=MAX_LEVEL_CHANGE, max_logit_shift=MAX_LOGIT_SHIFT, threads=THREADS,
        )
        predictions = {"original_lp": va["base"], "original_v5": va["v5"],
                       "correction_control": corrected["control"],
                       "correction_enhanced": corrected["enhanced"]}
        scores = score_predictions(va["truth"], predictions, units, va["starts"])
        scores["train_year"], scores["validation_year"] = train_year, valid_year
        all_scores.append(scores)
        fit_logs.append(fit_log.assign(train_year=train_year, validation_year=valid_year))
        coefficient_tables.append(coefs.assign(train_year=train_year, validation_year=valid_year))
        probe_predictions[valid_year] = corrected
        # Kun valideringsprediksjoner i vår nye mappe; aldri en submission.
        np.savez_compressed(OUT / f"validation_predictions_{valid_year}.npz",
            case_ids=va["case_ids"], correction_control=corrected["control"],
            correction_enhanced=corrected["enhanced"])
        print(f"{train_year} → {valid_year}: {time.perf_counter()-t0:.1f}s, inkludert evaluering")
    results = pd.concat(all_scores, ignore_index=True)
    results.to_csv(OUT / "metrics.csv", index=False)
    pd.concat(fit_logs, ignore_index=True).to_csv(OUT / "fit_diagnostics.csv", index=False)
    pd.concat(coefficient_tables, ignore_index=True).to_csv(OUT / "standardized_coefficients.csv", index=False)
    display(results.query("group == 'all'").pivot(
        index="model", columns="validation_year", values=["micro_auc","weekly_level_mae","brier"]).round(6))
else:
    print("RUN_SMALL_MODELS=False: ingen modell er trent.")


2020 → 2021: 1.4s, inkludert evaluering
2021 → 2022: 1.3s, inkludert evaluering


micro_auc           weekly_level_mae               brier  \
validation_year          2021      2022             2021      2022      2021   
model                                                                          
correction_control   0.993825  0.992415         0.035183  0.033806  0.030922   
correction_enhanced  0.993556  0.992387         0.035473  0.033567  0.031070   
original_lp          0.993805  0.992564         0.035226  0.033044  0.030839   
original_v5          0.985145  0.986819         0.061970  0.051129  0.049283   

                               
validation_year          2022  
model                          
correction_control   0.031648  
correction_enhanced  0.031524  
original_lp          0.031349  
original_v5          0.042713

## 4. Konklusjon — utforskende, ikke automatisk modellvalg

Enhanced må sammenlignes både med **original LP** og **kontrollkorreksjonen**.
Bedre ukentlig nivå-MAE er ikke tilstrekkelig hvis micro-AUC faller.

Vi viser også måneds-/generatorfordelingen. Ikke lag generator-spesifikke regler ved å plukke
vinnere på 2022. En liten positiv differanse er ikke dokumentert signifikans.
Rangeringen innen en uke/generator er bevart av logit-skiftet; forbedring må derfor komme
fra nivåjustering mellom uker/generatorer, ikke bedre intern timing.

En negativ test betyr at **denne korreksjonen** ikke hjalp; den beviser ikke at feature-ene
aldri kan hjelpe en annen modell. En positiv test må replikeres før den erstatter 0.99571-kandidaten.


In [6]:
decisions = []
if RUN_SMALL_MODELS:
    for train_year, valid_year in pairs:
        overall = results.query("validation_year == @valid_year and group == 'all'").set_index("model")
        enhanced = overall.loc["correction_enhanced"]
        gain_original = float(enhanced.micro_auc - overall.loc["original_lp", "micro_auc"])
        gain_control = float(enhanced.micro_auc - overall.loc["correction_control", "micro_auc"])
        level_gain = float(overall.loc["original_lp","weekly_level_mae"]-enhanced.weekly_level_mae)
        decisions.append(dict(train_year=train_year, validation_year=valid_year,
            auc_delta_vs_original=gain_original, auc_delta_vs_control=gain_control,
            level_mae_reduction_vs_original=level_gain))
    decision_table = pd.DataFrame(decisions)
    display(decision_table.round(7))
    for group in ("unit", "month"):
        detailed = results.query("validation_year == 2022 and group == @group").pivot(
            index="label", columns="model", values="micro_auc")
        detailed["enhanced_minus_original"] = detailed["correction_enhanced"]-detailed["original_lp"]
        detailed["enhanced_minus_control"] = detailed["correction_enhanced"]-detailed["correction_control"]
        display(detailed.sort_values("enhanced_minus_original").round(6))
    if all(d["auc_delta_vs_original"] > 0 and d["auc_delta_vs_control"] > 0 for d in decisions):
        conclusion = "LOVENDE PUNKTESTIMATER — utforskende, ikke statistisk bekreftet eller en ny submission."
    else:
        conclusion = "INGEN KONSISTENT EKSTRA AUC-GEVINST — behold original LP; se delresultatene."
    print(conclusion)
else:
    conclusion = "BARE DIAGNOSTIKK — ingen korreksjonsmodeller kjørt."


,train_year,validation_year,auc_delta_vs_original,auc_delta_vs_control,level_mae_reduction_vs_original
0,2020,2021,-0.000249,-0.000269,-0.000247
1,2021,2022,-0.000177,-0.000029,-0.000523


model,correction_control,correction_enhanced,original_lp,original_v5,enhanced_minus_original,enhanced_minus_control
label,,,,,,
Kjela_G1,0.987261,0.986025,0.987871,0.987606,-0.001846,-0.001235
Lio_G1,0.975935,0.974627,0.975722,0.962559,-0.001095,-0.001309
Songa_G1,0.994917,0.994861,0.995903,0.996025,-0.001042,-0.000056
Vesle Kjela_G1,0.995419,0.995271,0.995724,0.991814,-0.000453,-0.000147
Vinje_G1,0.993079,0.993079,0.993474,0.986367,-0.000396,-0.000001
Vinje_G2,0.989468,0.989760,0.989940,0.982011,-0.000180,0.000292
Byrte_G1,0.999452,0.999406,0.999507,0.998718,-0.000101,-0.000046
Vinje_G3,0.981839,0.981915,0.982007,0.974402,-0.000092,0.000076
Tokke_G1,0.997971,0.997956,0.998041,0.994766,-0.000085,-0.000016


model,correction_control,correction_enhanced,original_lp,original_v5,enhanced_minus_original,enhanced_minus_control
label,,,,,,
6,0.982519,0.981099,0.985558,0.975530,-0.004459,-0.001419
7,0.981380,0.981808,0.982545,0.974487,-0.000737,0.000428
8,0.996871,0.996952,0.997259,0.996091,-0.000307,0.000081
9,0.997651,0.997463,0.997764,0.993589,-0.000301,-0.000188
10,0.995369,0.995532,0.995727,0.987960,-0.000195,0.000163
3,0.995440,0.995445,0.995611,0.993231,-0.000167,0.000004
5,0.990779,0.990597,0.990753,0.983115,-0.000156,-0.000181
2,0.996936,0.996798,0.996933,0.994498,-0.000135,-0.000138
12,0.997702,0.997732,0.997776,0.994954,-0.000044,0.000030


INGEN KONSISTENT EKSTRA AUC-GEVINST — behold original LP; se delresultatene.


## 5. Sporbarhet og kontroll av originalfilene

Ingen testprediksjoner/Kaggle-filer produseres her. Hele modellen må senere kunne gjenskapes
fra rådata i en selvstendig kortnotebook hvis forsøket fører til en innlevering.
Denne notebooken er **ikke professor-klar**, fordi den bevisst gjenbruker artifactene fra 19.

Siste celle kontrollerer at innholdet i eksisterende CSV-er er uendret og lagrer filhashene
sammen med oppsett, input-identitet, kildekode og konklusjon i den nye resultatmappen.


In [7]:
changed = [name for name, old in ORIGINAL_CSV_HASHES.items()
           if not (ROOT / name).is_file() or file_digest(ROOT / name) != old]
if changed:
    raise RuntimeError(f"Originale CSV-er er endret siden starten (eventuelt av en annen prosess): {changed}")
settings = dict(alpha=RIDGE_ALPHA, shrinkage=SHRINKAGE,
    max_level_change=MAX_LEVEL_CHANGE, max_logit_shift=MAX_LOGIT_SHIFT,
    threads=THREADS, pairs=pairs, run_small_models=RUN_SMALL_MODELS)
notebook = ROOT / "notebooks/20_weekly_correction_probe.ipynb"
nb_sources = [c.get("source",[]) for c in json.loads(notebook.read_text())["cells"]]
manifest = dict(run_id=RUN_ID, settings=settings, provenance=provenance,
    helper_sha256=file_digest(ROOT / "src/sintef_weekly_probe.py"),
    notebook_sources_sha256=hashlib.sha256(json.dumps(nb_sources).encode()).hexdigest(),
    packages={p:importlib.metadata.version(p) for p in ("numpy","pandas","scipy","scikit-learn","threadpoolctl")},
    original_csv_sha256=ORIGINAL_CSV_HASHES, original_csv_unchanged=True,
    conclusion=conclusion, comparisons=decisions,
    elapsed_seconds=round(time.perf_counter()-STARTED,2))
(OUT / "probe_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Kontrollert: {len(ORIGINAL_CSV_HASHES)} originale CSV-er er uendret.")
print(conclusion)
print("Resultater:", OUT)
print("Ingen original submission er endret. Send tabellen og konklusjonen fra seksjon 4.")


Kontrollert: 248 originale CSV-er er uendret.
INGEN KONSISTENT EKSTRA AUC-GEVINST — behold original LP; se delresultatene.
Resultater: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/weekly_correction_probe/20260928T233806Z_3aa70e0a
Ingen original submission er endret. Send tabellen og konklusjonen fra seksjon 4.
